In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from statsmodels.tsa.statespace.sarimax import SARIMAX
from scipy.stats import norm

In [ ]:
H = 7                                                   # dias a predecir desde el origen
QL = np.array([.02, .05, .10, .25, .50, .75, .90, .95, .98])   # cuantiles
CAL = ["es_festivo", "es_puente", "es_vispera_festivo", "es_post_festivo", "es_quincena",
       "es_vacaciones_escolares", "es_semana_santa", "es_temporada_navidad", "es_temporada_fin_anio",
       "es_regreso_colegio", "es_covid_parcial"]
RAROS = ["fut_m_feria_d0_4", "fut_m_noche_festiva", "fut_m_dia_siguiente",
         "ext_m_catastrofe_sem1", "ext_m_catastrofe_sem2_3", "ext_m_paro_2021"]
SARIMAX_COLS = RAROS + ["fut_m_fiesta_lejana"]
PARAM_LGB = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=30,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1, n_jobs=2)

In [ ]:
def calendario(idx, cal):
    X = pd.get_dummies(pd.Index(idx.dayofweek), prefix="dow", drop_first=True).set_index(idx).astype(float)
    X = X.join(cal.reindex(idx).fillna(0))
    doy = 2 * np.pi * idx.dayofyear.values / 365.25
    for k in (1, 2, 3):
        X[f"four_s{k}"] = np.sin(k * doy); X[f"four_c{k}"] = np.cos(k * doy)
    return X


def rasgos(y, cal, h, PAS=None, FUT=None):
    """Entradas para predecir t desde el origen o = t - h. PAS: contexto conocido en el origen; FUT: conocido por adelantado."""
    ly = np.log(y)
    nivel = ly.rolling(28, min_periods=20).mean()
    norma_dow = ly.groupby(ly.index.dayofweek).transform(lambda s: s.shift(1).rolling(4, min_periods=2).mean())
    X = calendario(y.index, cal)
    X["nivel"] = nivel.shift(h)
    X["mem_semana"] = (ly.rolling(7, min_periods=5).mean() - nivel).shift(h)
    X["mem_ultimo"] = (ly - norma_dow).shift(h)
    X["mem_mismo_dia"] = ly.shift(7 * -(-h // 7)) - nivel.shift(h)
    if PAS is not None: X = X.join(PAS.shift(h).reindex(X.index))
    if FUT is not None: X = X.join(FUT.reindex(X.index))
    return X

In [ ]:
class ModeloSARIMAX:
    """SARIMA(1,0,1)(0,1,1)7 sobre log(y) con calendario, estacion y mecanismos como exogenas."""
    def __init__(self, y, cal, PAS=None, FUT=None):
        self.y = y
        E = calendario(cal.index, cal); E = E[[c for c in E if not c.startswith("dow_")]]
        if FUT is not None: E = E.join(FUT[[c for c in FUT if c in SARIMAX_COLS]])
        self.PAS = None if PAS is None else PAS[[c for c in PAS if c in SARIMAX_COLS]]
        if self.PAS is not None: E = E.join(self.PAS.shift(1).fillna(0))
        self.E = E.fillna(0)

    def _modelo(self, o):
        return SARIMAX(np.log(self.y.loc[:o]), exog=self.E.loc[:o, self.cols], order=(1, 0, 1), seasonal_order=(0, 1, 1, 7))

    def ajustar(self, o):
        Ez = self.E.loc[:o]; self.cols = Ez.columns[Ez.std() > 0]
        self.params = self._modelo(o).fit(disp=0, maxiter=100).params

    def predecir(self, o, n=H):
        F = self.E.loc[o + pd.Timedelta(days=1):o + pd.Timedelta(days=n)].copy()
        if self.PAS is not None: F[self.PAS.columns] = self.PAS.loc[o].values
        f = self._modelo(o).filter(self.params).get_forecast(n, exog=F[self.cols])
        return np.exp(f.predicted_mean.values[:, None] + f.se_mean.values[:, None] * norm.ppf(QL)[None, :])

In [ ]:
class ModeloLightGBM:
    """Un LightGBM por horizonte y por cuantil sobre log(y) - nivel; los episodios raros como desplazamiento lineal."""
    def __init__(self, y, cal, PAS=None, FUT=None):
        self.y = y
        self.X = {h: rasgos(y, cal.reindex(y.index), h, PAS, FUT) for h in range(1, H + 1)}

    def _datos(self, h, o):
        X = self.X[h].loc[:o]; yy = self.y.loc[:o]
        ok = X[[c for c in X if not c.startswith("ext_g_")]].notna().all(axis=1) & yy.notna()   # ext_g_: compuerta (NaN = apagada)
        return X[ok], yy[ok]

    def ajustar(self, o):
        self.cols, self.m = {}, {}
        for h in range(1, H + 1):
            X, yy = self._datos(h, o)
            self.cols[h] = X.columns[X.std() > 0]
            X = X[self.cols[h]]
            Z = X.drop(columns="nivel"); r = np.log(yy) - X.nivel
            rar = [c for c in RAROS if c in Z]
            b = np.zeros(len(rar))
            if rar:
                ctrl = [c for c in Z if c.startswith(("dow_", "es_"))]
                A = np.c_[np.ones(len(Z)), Z[ctrl].values, Z[rar].values]
                b = np.linalg.lstsq(A, r.values, rcond=None)[0][-len(rar):]
            off = Z[rar].values @ b if rar else 0.0
            ms = [lgb.LGBMRegressor(objective="quantile", alpha=q, **PARAM_LGB).fit(Z, r - off) for q in QL]
            self.m[h] = (ms, rar, b)

    def predecir(self, o):
        filas = []
        for h in range(1, H + 1):
            x = self.X[h].loc[[o + pd.Timedelta(days=h)], self.cols[h]]
            ms, rar, b = self.m[h]; z = x.drop(columns="nivel")
            off = float(z[rar].values[0] @ b) if rar else 0.0
            filas.append(np.exp(np.sort([mm.predict(z)[0] for mm in ms]) + x.nivel.values[0] + off))
        return np.vstack(filas)

In [ ]:
def combinar(*Q):
    """Combinacion: promedio de los cuantiles de cada modelo, ordenados."""
    return np.sort(np.mean(Q, axis=0), axis=1)

In [ ]:
# y: serie diaria de triages (indice = fecha); cal: columnas CAL por fecha; PAS/FUT: mecanismos (opcionales)
# En el proyecto el entrenamiento excluye la fase fuerte de la COVID: y.loc["2020-03-15":"2020-08-31"] = np.nan
y = pd.read_parquet("triage_diario.parquet").assign(dia=lambda d: pd.to_datetime(d.dia)).set_index("dia")["triages"].astype(float)
y = y.reindex(pd.date_range(y.index[0], y.index[-1] + pd.Timedelta(days=H)))
cal = pd.read_csv("calendario_Colombia.csv", parse_dates=["fecha"]).set_index("fecha").reindex(y.index)
cal = cal.reindex(columns=CAL).apply(pd.to_numeric, errors="coerce").fillna(0)
PAS, FUT = None, None

o = pd.Timestamp("2026-09-13")                          # origen: ultimo dia conocido
s = ModeloSARIMAX(y, cal, PAS, FUT); s.ajustar(o)
g = ModeloLightGBM(y, cal, PAS, FUT); g.ajustar(o)
Q = combinar(s.predecir(o), g.predecir(o))
pd.DataFrame(Q, columns=[f"q{int(q*100):02d}" for q in QL], index=pd.date_range(o + pd.Timedelta(days=1), periods=H)).round(1)